In [1]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils

In [2]:
df_se = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/se_bn_no_context.csv")

In [3]:
df_se = df_se.drop(["Unnamed: 0"], axis = 1)

In [4]:
df_se

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,362059.214096,25.02,79.7,6.030000,5.65,19.26,43.224,5.900063,24.205811,3.214195,9.42,7.6,10.5,0.000006,276.174863,3.554298
1,367898.323980,24.15,80.0,6.020000,5.63,20.21,43.922,5.218714,24.785451,2.032230,8.05,8.0,13.5,0.000006,276.464208,5.290971
2,357492.674906,22.33,77.9,6.040000,5.69,25.02,47.024,4.958800,26.132116,-0.614954,2.06,9.1,14.4,0.000005,235.053763,19.503994
3,378703.698372,23.86,77.7,6.200000,5.71,24.76,46.099,4.661454,25.187785,-0.240938,0.13,8.0,14.8,0.000006,234.115139,18.684230
4,392078.684225,23.73,79.0,6.250000,5.66,22.78,47.632,4.436392,25.092481,0.015331,1.22,7.1,15.4,0.000005,212.063492,33.708495
5,391255.979647,23.11,79.1,5.970000,5.58,23.61,49.403,4.367928,25.878585,-0.914026,0.78,7.7,15.2,0.000005,240.336134,11.559979
6,396337.894577,21.98,79.4,5.730000,5.52,23.52,50.153,3.831176,26.318281,-1.371356,0.25,9.3,16.0,0.000005,242.916667,26.080148
7,408791.051257,21.61,79.6,5.670000,5.50,22.82,51.151,3.093335,26.275487,-1.129253,-1.05,10.6,15.6,0.000004,204.536082,35.538469
8,431846.583992,22.28,80.1,5.600000,5.36,20.36,52.220,3.259861,25.892770,0.377740,1.95,10.6,16.3,0.000004,207.959184,8.000893
9,443685.809403,21.96,80.8,5.700000,5.30,18.90,52.597,2.557094,26.439038,1.576112,3.01,14.5,16.2,0.000004,204.838710,5.269573


In [5]:
se = pd.DataFrame()
se = df_se.copy()
se['QOR_2'] = se['QOR'].shift(2)
se['QOA_2'] = se['QOA'].shift(2)
se['GGFC_3'] = se['GGFC'].shift(3)


In [6]:
se = se.dropna()

In [7]:
sm = StructureModel()

In [8]:
sm.add_edges_from([
    ('EUPC', 'QOA'),
     ('EUPC', 'FDI'),
     ('QOR_2', 'FDI'),
     ('QOR_2', 'GDPC'),
    ('GGFC_3', 'EUPC'),
    
])

In [9]:
sm.edges

OutEdgeView([('EUPC', 'QOA'), ('EUPC', 'FDI'), ('QOR_2', 'FDI'), ('QOR_2', 'GDPC'), ('GGFC_3', 'EUPC')])

In [10]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_se_1.html")

Graphs/fully_connected_se_1.html


In [11]:
bn = BayesianNetwork(sm)

In [12]:
discretised_se = pd.DataFrame(index=se.index)

for col in se.columns:
    no_unique = se[col].nunique()
    
    if no_unique <= 1:
        discretised_se[col] = 0
    else:
        try:
            discretised_se[col] = pd.qcut(
                se[col], 
                q=min(3, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_se[col] = se[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_se)

Discretised data:
    GDPC  VABI  EMP  QOA  QOR  YUA  REST  EBSG  GGFC  LBGD  FDI  PC  ARP  \
3      0     2    0    2    1    2     0     2     0     1    0   1    0   
4      0     2    0    2    1    1     0     2     0     1    1   0    0   
5      0     2    0    2    1    2     0     2     1     0    0   0    0   
6      0     0    0    0    0    1     0     1     2     0    0   1    1   
7      0     0    0    0    0    1     0     0     2     0    0   1    0   
8      1     1    1    0    0    0     1     1     1     1    1   1    2   
9      1     0    1    0    0    0     1     0     2     2    1   2    2   
10     1     0    2    1    0    0     1     0     1     2    2   2    1   
11     1     1    2    1    1    0     1     0     1     2    0   2    2   
12     2     1    2    0    0    0     2     1     0     2    1   1    2   
13     2     0    1    1    0    2     2     1     2     0    2   0    1   
14     2     1    1    1    0    2     2     2     0     0    2   0   

In [13]:
discretised_se = discretised_se.reset_index(drop=True)
bn.fit_node_states(discretised_se)
baseline_auc = utils.get_avg_auc_all_info(discretised_se, bn)
print(f"Baseline AUC: {baseline_auc}")

Processing fold 0 using 7 cores takes 6.585052251815796 seconds
Processing fold 1 using 7 cores takes 5.88605809211731 seconds
Processing fold 2 using 7 cores takes 5.7013161182403564 seconds
Processing fold 3 using 7 cores takes 5.55886697769165 seconds
Processing fold 4 using 7 cores takes 5.73737096786499 seconds
Baseline AUC: 0.6201388888888889


In [14]:
edges_to_add = [('LV', 'EUPC'), ('LV', 'QOA')]
edges_to_remove = [('EUPC', 'QOA')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [15]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_se_2.html")

Graphs/node_added_se_2.html


In [16]:
discretised_se['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_se, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EUPC' and 'QOA': {proposed_auc}")

Processing fold 0 using 7 cores takes 6.4692511558532715 seconds
Processing fold 1 using 7 cores takes 5.559870004653931 seconds
Processing fold 2 using 7 cores takes 5.6647629737854 seconds
Processing fold 3 using 7 cores takes 5.884284973144531 seconds
Processing fold 4 using 7 cores takes 5.666372060775757 seconds
AUC from adding LV between 'EUPC' and 'QOA': 0.6064814814814815


In [17]:
edges_to_add = [('LV', 'EUPC'), ('LV', 'FDI')]
edges_to_remove = [('EUPC', 'FDI')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [18]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_se_3.html")

Graphs/node_added_se_3.html


In [19]:
discretised_se['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_se, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EUPC' and 'FDI': {proposed_auc}")

Processing fold 0 using 7 cores takes 6.457294940948486 seconds
Processing fold 1 using 7 cores takes 5.7443201541900635 seconds
Processing fold 2 using 7 cores takes 5.679301977157593 seconds
Processing fold 3 using 7 cores takes 5.699502944946289 seconds
Processing fold 4 using 7 cores takes 5.611788988113403 seconds
AUC from adding LV between 'EUPC' and 'FDI': 0.608101851851852


In [20]:
edges_to_add = [('LV', 'GGFC_3'), ('LV', 'EUPC')]
edges_to_remove = [('GGFC_3', 'EUPC')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [21]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_se_3.html")

Graphs/node_added_se_3.html


In [22]:
discretised_se['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_se, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'GGFC_3' and 'EUPC': {proposed_auc}")

Processing fold 0 using 7 cores takes 6.251547813415527 seconds
Processing fold 1 using 7 cores takes 5.785547971725464 seconds
Processing fold 2 using 7 cores takes 5.71160888671875 seconds
Processing fold 3 using 7 cores takes 5.7392258644104 seconds
Processing fold 4 using 7 cores takes 5.684161186218262 seconds
AUC from adding LV between 'GGFC_3' and 'EUPC': 0.7493055555555554


In [43]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils

In [44]:
df_se_2 = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/se_bn_no_context.csv")

In [45]:
df_se_2 = df_se_2.drop(["Unnamed: 0"], axis = 1)

In [46]:
df_se_2

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,362059.214096,25.02,79.7,6.030000,5.65,19.26,43.224,5.900063,24.205811,3.214195,9.42,7.6,10.5,0.000006,276.174863,3.554298
1,367898.323980,24.15,80.0,6.020000,5.63,20.21,43.922,5.218714,24.785451,2.032230,8.05,8.0,13.5,0.000006,276.464208,5.290971
2,357492.674906,22.33,77.9,6.040000,5.69,25.02,47.024,4.958800,26.132116,-0.614954,2.06,9.1,14.4,0.000005,235.053763,19.503994
3,378703.698372,23.86,77.7,6.200000,5.71,24.76,46.099,4.661454,25.187785,-0.240938,0.13,8.0,14.8,0.000006,234.115139,18.684230
4,392078.684225,23.73,79.0,6.250000,5.66,22.78,47.632,4.436392,25.092481,0.015331,1.22,7.1,15.4,0.000005,212.063492,33.708495
5,391255.979647,23.11,79.1,5.970000,5.58,23.61,49.403,4.367928,25.878585,-0.914026,0.78,7.7,15.2,0.000005,240.336134,11.559979
6,396337.894577,21.98,79.4,5.730000,5.52,23.52,50.153,3.831176,26.318281,-1.371356,0.25,9.3,16.0,0.000005,242.916667,26.080148
7,408791.051257,21.61,79.6,5.670000,5.50,22.82,51.151,3.093335,26.275487,-1.129253,-1.05,10.6,15.6,0.000004,204.536082,35.538469
8,431846.583992,22.28,80.1,5.600000,5.36,20.36,52.220,3.259861,25.892770,0.377740,1.95,10.6,16.3,0.000004,207.959184,8.000893
9,443685.809403,21.96,80.8,5.700000,5.30,18.90,52.597,2.557094,26.439038,1.576112,3.01,14.5,16.2,0.000004,204.838710,5.269573


In [47]:
se_2 = pd.DataFrame()
se_2 = df_se_2.copy()
se_2['EUPC_2'] = se_2['EUPC'].shift(2)

In [48]:
se_2 = se_2.dropna()

In [49]:
sm2 = StructureModel()

In [50]:
sm2.add_edges_from([
    ('EUPC_2', 'QOR'),
    ('EUPC_2', 'EBSG'),
    ('QOR', 'ARP'),
    ('GDPC', 'ARP')
])

In [51]:
sm2.edges

OutEdgeView([('EUPC_2', 'QOR'), ('EUPC_2', 'EBSG'), ('QOR', 'ARP'), ('GDPC', 'ARP')])

In [52]:
viz = plot_structure(
    sm2,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_se_3.html")

Graphs/fully_connected_se_3.html


In [53]:
bn2 = BayesianNetwork(sm2)

In [54]:
discretised_se = pd.DataFrame(index=se_2.index)

for col in se_2.columns:
    no_unique = se_2[col].nunique()
    
    if no_unique <= 1:
        discretised_se[col] = 0
    else:
        try:
            discretised_se[col] = pd.qcut(
                se_2[col], 
                q=min(3, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_se[col] = se_2[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_se)

Discretised data:
    GDPC  VABI  EMP  QOA  QOR  YUA  REST  EBSG  GGFC  LBGD  FDI  PC  ARP  \
2      0     1    0    2    2    2     0     2     1     1    1   1    0   
3      0     2    0    2    2    2     0     2     0     1    0   1    0   
4      0     2    0    2    2    1     0     2     0     1    1   0    0   
5      0     2    0    2    2    2     0     2     1     0    0   0    0   
6      0     0    0    0    1    1     0     1     2     0    0   1    1   
7      1     0    1    0    1    1     1     0     2     0    0   2    0   
8      1     1    1    0    1    0     1     1     1     1    1   2    2   
9      1     0    2    0    0    0     1     0     2     2    2   2    2   
10     1     0    2    1    1    0     1     0     2     2    2   2    1   
11     2     1    2    1    2    0     2     0     1     2    0   2    2   
12     2     1    2    0    0    0     2     1     0     2    1   1    2   
13     2     0    1    1    0    2     2     1     2     0    2   0   

In [55]:
discretised_se = discretised_se.reset_index(drop=True)
bn2.fit_node_states(discretised_se)
baseline_auc = utils.get_avg_auc_all_info(discretised_se, bn2)
print(f"Baseline AUC: {baseline_auc}")

Processing fold 0 using 7 cores takes 5.810550212860107 seconds
Processing fold 1 using 7 cores takes 5.264471054077148 seconds
Processing fold 2 using 7 cores takes 5.286329030990601 seconds
Processing fold 3 using 7 cores takes 5.278055906295776 seconds
Processing fold 4 using 7 cores takes 5.279097080230713 seconds
Baseline AUC: 0.5741666666666666


In [56]:
edges_to_add = [('LV', 'EUPC_2'), ('LV', 'QOR')]
edges_to_remove = [('EUPC_2', 'QOR')]

bn_with_lv = copy.deepcopy(bn2)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [57]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_se_7.html")

Graphs/node_added_se_7.html


In [58]:
discretised_se['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_se, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EUPC_2' and 'QOR': {proposed_auc}")

Processing fold 0 using 7 cores takes 5.838076829910278 seconds
Processing fold 1 using 7 cores takes 5.611438989639282 seconds
Processing fold 2 using 7 cores takes 5.362547874450684 seconds
Processing fold 3 using 7 cores takes 5.388333797454834 seconds
Processing fold 4 using 7 cores takes 5.6353888511657715 seconds
AUC from adding LV between 'EUPC_2' and 'QOR': 0.6661111111111111


In [59]:
edges_to_add = [('LV', 'EUPC_2'), ('LV', 'EBSG')]
edges_to_remove = [('EUPC_2', 'EBSG')]

bn_with_lv = copy.deepcopy(bn2)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [60]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_se_8.html")

Graphs/node_added_se_8.html


In [61]:
discretised_se['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_se, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EUPC_2' and 'EBSG': {proposed_auc}")

Processing fold 0 using 7 cores takes 5.822542905807495 seconds
Processing fold 1 using 7 cores takes 5.482858896255493 seconds
Processing fold 2 using 7 cores takes 5.500271797180176 seconds
Processing fold 3 using 7 cores takes 5.576732873916626 seconds
Processing fold 4 using 7 cores takes 5.490065813064575 seconds
AUC from adding LV between 'EUPC_2' and 'EBSG': 0.7200000000000001


In [62]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils

In [63]:
df_se_2 = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/se_bn_no_context.csv")

In [64]:
df_se_2 = df_se_2.drop(["Unnamed: 0"], axis = 1)

In [65]:
df_se_2

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,362059.214096,25.02,79.7,6.030000,5.65,19.26,43.224,5.900063,24.205811,3.214195,9.42,7.6,10.5,0.000006,276.174863,3.554298
1,367898.323980,24.15,80.0,6.020000,5.63,20.21,43.922,5.218714,24.785451,2.032230,8.05,8.0,13.5,0.000006,276.464208,5.290971
2,357492.674906,22.33,77.9,6.040000,5.69,25.02,47.024,4.958800,26.132116,-0.614954,2.06,9.1,14.4,0.000005,235.053763,19.503994
3,378703.698372,23.86,77.7,6.200000,5.71,24.76,46.099,4.661454,25.187785,-0.240938,0.13,8.0,14.8,0.000006,234.115139,18.684230
4,392078.684225,23.73,79.0,6.250000,5.66,22.78,47.632,4.436392,25.092481,0.015331,1.22,7.1,15.4,0.000005,212.063492,33.708495
5,391255.979647,23.11,79.1,5.970000,5.58,23.61,49.403,4.367928,25.878585,-0.914026,0.78,7.7,15.2,0.000005,240.336134,11.559979
6,396337.894577,21.98,79.4,5.730000,5.52,23.52,50.153,3.831176,26.318281,-1.371356,0.25,9.3,16.0,0.000005,242.916667,26.080148
7,408791.051257,21.61,79.6,5.670000,5.50,22.82,51.151,3.093335,26.275487,-1.129253,-1.05,10.6,15.6,0.000004,204.536082,35.538469
8,431846.583992,22.28,80.1,5.600000,5.36,20.36,52.220,3.259861,25.892770,0.377740,1.95,10.6,16.3,0.000004,207.959184,8.000893
9,443685.809403,21.96,80.8,5.700000,5.30,18.90,52.597,2.557094,26.439038,1.576112,3.01,14.5,16.2,0.000004,204.838710,5.269573


In [66]:
se_2 = pd.DataFrame()
se_2 = df_se_2.copy()
se_2['EUPC_1'] = se_2['EUPC'].shift(1)

In [67]:
se_2 = se_2.dropna()

In [68]:
sm2 = StructureModel()

In [69]:
sm2.add_edges_from([
    ('EUPC_1', 'PC')
    
])

In [70]:
sm2.edges

OutEdgeView([('EUPC_1', 'PC')])

In [71]:
viz = plot_structure(
    sm2,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_se_1.html")

Graphs/fully_connected_se_1.html


In [72]:
bn2 = BayesianNetwork(sm2)

In [73]:
discretised_se = pd.DataFrame(index=se_2.index)

for col in se_2.columns:
    no_unique = se_2[col].nunique()
    
    if no_unique <= 1:
        discretised_se[col] = 0
    else:
        try:
            discretised_se[col] = pd.qcut(
                se_2[col], 
                q=min(3, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_se[col] = se_2[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_se)

Discretised data:
    GDPC  VABI  EMP  QOA  QOR  YUA  REST  EBSG  GGFC  LBGD  FDI  PC  ARP  \
1      0     2    1    2    2    0     0     2     0     2    2   1    0   
2      0     1    0    2    2    2     0     2     1     1    1   1    0   
3      0     2    0    2    2    2     0     2     0     1    0   1    0   
4      0     2    0    2    2    1     0     2     0     1    1   0    0   
5      0     2    0    2    1    2     0     2     1     0    0   0    0   
6      1     0    0    0    1    1     1     1     2     0    0   1    1   
7      1     0    1    0    1    1     1     0     2     0    0   2    1   
8      1     1    1    0    1    1     1     1     1     1    1   2    2   
9      1     0    2    0    0    0     1     0     2     2    1   2    2   
10     1     0    2    1    1    0     1     0     2     2    2   2    1   
11     2     1    2    1    2    0     2     0     1     2    0   2    2   
12     2     1    2    0    0    0     2     1     1     2    1   1   

In [74]:
discretised_se = discretised_se.reset_index(drop=True)
bn2.fit_node_states(discretised_se)
baseline_auc = utils.get_avg_auc_all_info(discretised_se, bn2)
print(f"Baseline AUC: {baseline_auc}")

Processing fold 0 using 7 cores takes 4.420755863189697 seconds
Processing fold 1 using 7 cores takes 4.459047079086304 seconds
Processing fold 2 using 7 cores takes 4.424137115478516 seconds
Processing fold 3 using 7 cores takes 4.461359024047852 seconds
Processing fold 4 using 7 cores takes 4.565962076187134 seconds
Baseline AUC: 0.4805555555555555


In [75]:
edges_to_add = [('LV', 'EUPC_1'), ('LV', 'PC')]
edges_to_remove = [('EUPC_1', 'PC')]

bn_with_lv = copy.deepcopy(bn2)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [76]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_se_9.html")

Graphs/node_added_se_9.html


In [77]:
discretised_se['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_se, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EUPC_1' and 'PC': {proposed_auc}")

Processing fold 0 using 7 cores takes 4.978957891464233 seconds
Processing fold 1 using 7 cores takes 5.211888074874878 seconds
Processing fold 2 using 7 cores takes 4.49842381477356 seconds
Processing fold 3 using 7 cores takes 4.410171985626221 seconds
Processing fold 4 using 7 cores takes 4.597265720367432 seconds
AUC from adding LV between 'EUPC_1' and 'PC': 0.4888888888888888


In [78]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils

In [79]:
df_se_2 = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/se_bn_no_context.csv")

In [80]:
df_se_2 = df_se_2.drop(["Unnamed: 0"], axis = 1)

In [81]:
df_se_2

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,362059.214096,25.02,79.7,6.030000,5.65,19.26,43.224,5.900063,24.205811,3.214195,9.42,7.6,10.5,0.000006,276.174863,3.554298
1,367898.323980,24.15,80.0,6.020000,5.63,20.21,43.922,5.218714,24.785451,2.032230,8.05,8.0,13.5,0.000006,276.464208,5.290971
2,357492.674906,22.33,77.9,6.040000,5.69,25.02,47.024,4.958800,26.132116,-0.614954,2.06,9.1,14.4,0.000005,235.053763,19.503994
3,378703.698372,23.86,77.7,6.200000,5.71,24.76,46.099,4.661454,25.187785,-0.240938,0.13,8.0,14.8,0.000006,234.115139,18.684230
4,392078.684225,23.73,79.0,6.250000,5.66,22.78,47.632,4.436392,25.092481,0.015331,1.22,7.1,15.4,0.000005,212.063492,33.708495
5,391255.979647,23.11,79.1,5.970000,5.58,23.61,49.403,4.367928,25.878585,-0.914026,0.78,7.7,15.2,0.000005,240.336134,11.559979
6,396337.894577,21.98,79.4,5.730000,5.52,23.52,50.153,3.831176,26.318281,-1.371356,0.25,9.3,16.0,0.000005,242.916667,26.080148
7,408791.051257,21.61,79.6,5.670000,5.50,22.82,51.151,3.093335,26.275487,-1.129253,-1.05,10.6,15.6,0.000004,204.536082,35.538469
8,431846.583992,22.28,80.1,5.600000,5.36,20.36,52.220,3.259861,25.892770,0.377740,1.95,10.6,16.3,0.000004,207.959184,8.000893
9,443685.809403,21.96,80.8,5.700000,5.30,18.90,52.597,2.557094,26.439038,1.576112,3.01,14.5,16.2,0.000004,204.838710,5.269573


In [82]:
se_2 = pd.DataFrame()
se_2 = df_se_2.copy()
se_2['QOA_2'] = se_2['QOA'].shift(2)

In [83]:
se_2 = se_2.dropna()

In [84]:
sm2 = StructureModel()

In [85]:
sm2.add_edges_from([
    ('QOA_2', 'EUPC'),
])

In [86]:
sm2.edges

OutEdgeView([('QOA_2', 'EUPC')])

In [87]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_se_1.html")

Graphs/fully_connected_se_1.html


In [88]:
bn2 = BayesianNetwork(sm2)

In [89]:
discretised_se = pd.DataFrame(index=se_2.index)

for col in se_2.columns:
    no_unique = se_2[col].nunique()
    
    if no_unique <= 1:
        discretised_se[col] = 0
    else:
        try:
            discretised_se[col] = pd.qcut(
                se_2[col], 
                q=min(3, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_se[col] = se_2[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_se)

Discretised data:
    GDPC  VABI  EMP  QOA  QOR  YUA  REST  EBSG  GGFC  LBGD  FDI  PC  ARP  \
2      0     1    0    2    2    2     0     2     1     1    1   1    0   
3      0     2    0    2    2    2     0     2     0     1    0   1    0   
4      0     2    0    2    2    1     0     2     0     1    1   0    0   
5      0     2    0    2    2    2     0     2     1     0    0   0    0   
6      0     0    0    0    1    1     0     1     2     0    0   1    1   
7      1     0    1    0    1    1     1     0     2     0    0   2    0   
8      1     1    1    0    1    0     1     1     1     1    1   2    2   
9      1     0    2    0    0    0     1     0     2     2    2   2    2   
10     1     0    2    1    1    0     1     0     2     2    2   2    1   
11     2     1    2    1    2    0     2     0     1     2    0   2    2   
12     2     1    2    0    0    0     2     1     0     2    1   1    2   
13     2     0    1    1    0    2     2     1     2     0    2   0   

In [90]:
discretised_se = discretised_se.reset_index(drop=True)
bn2.fit_node_states(discretised_se)
baseline_auc = utils.get_avg_auc_all_info(discretised_se, bn2)
print(f"Baseline AUC: {baseline_auc}")

Processing fold 0 using 7 cores takes 4.90828800201416 seconds
Processing fold 1 using 7 cores takes 4.448734998703003 seconds
Processing fold 2 using 7 cores takes 4.278395891189575 seconds
Processing fold 3 using 7 cores takes 4.407802104949951 seconds
Processing fold 4 using 7 cores takes 4.358614921569824 seconds
Baseline AUC: 0.34236111111111106


In [91]:
edges_to_add = [('LV', 'QOA_2'), ('LV', 'EUPC')]
edges_to_remove = [('QOA_2', 'EUPC')]

bn_with_lv = copy.deepcopy(bn2)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [92]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_se_10.html")

Graphs/node_added_se_10.html


In [93]:
discretised_se['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_se, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'QOA_2' and 'EUPC': {proposed_auc}")

Processing fold 0 using 7 cores takes 4.67192816734314 seconds
Processing fold 1 using 7 cores takes 4.543488025665283 seconds
Processing fold 2 using 7 cores takes 4.499023914337158 seconds
Processing fold 3 using 7 cores takes 4.526490926742554 seconds
Processing fold 4 using 7 cores takes 4.483291149139404 seconds
AUC from adding LV between 'QOA_2' and 'EUPC': 0.4027777777777778
